# Mamba-2、Jamba 与混合架构

本 Notebook 介绍 Mamba 的高级变体和 Mamba+Transformer 混合架构。

## 目录
1. [Mamba-2: 结构化SSM与注意力的统一](#1-mamba-2)
2. [Jamba: Mamba + Transformer 混合架构](#2-jamba)
3. [RWKV: 线性 RNN](#3-rwkv)
4. [横向对比实验](#4-对比实验)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

---
## 1. Mamba-2: 结构化 SSM 与注意力的统一

Mamba-2 (Gu & Dao, 2024) 的核心贡献：

**结构化注意力** (Structured State Space Duality, SSD)
- 发现 SSM 可以看作一种特殊的结构化注意力
- SSM 矩阵是一个「半可分离矩阵」
- 可以用矩阵分解加速计算

关键改进：
1. **块对角 + 状态共享**: 更高效的并行计算
2. **更大的状态维度**: 在相同计算预算下表现更好
3. **硬件优化**: 更好的 GPU 利用率

In [ ]:
class Mamba2SSM(nn.Module):
    """Mamba-2 风格的 SSM (简化教学版本)
    
    核心变化:
    1. A 是标量 (而非矩阵), 即每个通道共享一个 A
    2. 使用块对角结构提高并行度
    3. 合并 BC 投影
    """
    def __init__(self, d_model, d_state=64, n_heads=16, chunk_size=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.chunk_size = chunk_size

        # A 参数: 每个head一个标量
        self.A_param = nn.Parameter(torch.randn(n_heads) * 0.5)
        self.D = nn.Parameter(torch.ones(n_heads))

        # 合并的 BC 投影
        self.BC_proj = nn.Linear(d_model, 2 * n_heads * d_state, bias=False)
        # dt 投影
        self.dt_proj = nn.Linear(d_model, n_heads, bias=True)

    def forward(self, x):
        """x: [B, L, D]"""
        B_batch, L, D = x.shape
        H = self.n_heads
        N = self.d_state
        P = self.d_head

        # 投影得到 B, C
        BC = self.BC_proj(x)  # [B, L, 2*H*N]
        B, C = BC.split([H * N, H * N], dim=-1)
        B = B.view(B_batch, L, H, N)   # [B, L, H, N]
        C = C.view(B_batch, L, H, N)   # [B, L, H, N]

        # dt
        dt = F.softplus(self.dt_proj(x))  # [B, L, H]

        # 离散化 A
        A = -torch.exp(self.A_param)  # [H] 保证负值

        # 递推 (教学用朴素实现)
        x_heads = x.view(B_batch, L, H, P)
        h = torch.zeros(B_batch, H, N, device=x.device)
        ys = []
        for t in range(L):
            dA = torch.exp(dt[:, t].unsqueeze(-1) * A.unsqueeze(0))  # [B, H, 1] * [1, H] -> [B, H, 1]
            dB = dt[:, t].unsqueeze(-1) * B[:, t]  # [B, H, N]
            h = h * dA + dB * x_heads[:, t].unsqueeze(-1)  # [B, H, N] * [B, H, 1] + [B, H, N] * [B, H, 1]
            y = torch.einsum('bhn,bhn->bh', h, C[:, t])  # [B, H]
            ys.append(y)

        y = torch.stack(ys, dim=1)  # [B, L, H]
        y = y + self.D * x_heads.mean(dim=-1)  # D residual
        y = y.view(B_batch, L, -1)  # [B, L, D]
        return y


# 验证
m2_ssm = Mamba2SSM(d_model=64, d_state=16, n_heads=4).to(device)
x = torch.randn(2, 32, 64, device=device)
out = m2_ssm(x)
print(f'Mamba-2 SSM: {x.shape} -> {out.shape}')

In [ ]:
# 可视化 SSM 矩阵结构
# SSM 的 Y = SSM(A, B, C) * X，其中 SSM 矩阵是半可分离的
def visualize_ssm_matrix(A_scalar, N, L):
    """可视化 SSM 矩阵的结构"""
    A = A_scalar * torch.ones(N)
    B = torch.ones(N) * 0.5
    C = torch.ones(N) * 0.5

    # 构造 SSM 矩阵: M[i,j] = C^T * A^(i-j) * B for i >= j
    M = torch.zeros(L, L)
    for i in range(L):
        for j in range(i + 1):
            power = i - j
            M[i, j] = (C * (A ** power) * B).sum().item()

    return M

# 对比标准注意力矩阵和 SSM 矩阵
L = 32
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 标准注意力: 全连接
attn = torch.ones(L, L)
axes[0].imshow(attn.numpy(), cmap='Blues')
axes[0].set_title('Standard Attention\n(full matrix)')

# SSM 矩阵
ssm_mat = visualize_ssm_matrix(-0.5, N=4, L=L)
axes[1].imshow(ssm_mat.numpy(), cmap='Blues')
axes[1].set_title('SSM Matrix\n(lower-triangular, decay)')

# 注意: 颜色越深 = 值越大
ssm_mat2 = visualize_ssm_matrix(-2.0, N=4, L=L)
axes[2].imshow(ssm_mat2.numpy(), cmap='Blues')
axes[2].set_title('SSM Matrix (larger |A|)\n(faster decay)')

plt.tight_layout()
plt.show()
print('SSM 矩阵是下三角的（因果性），且值随距离衰减（类似注意力中的局部偏好）')

---
## 2. Jamba: Mamba + Transformer 混合架构

Jamba (AI21 Labs, 2024) 将 Mamba 层和 Transformer 注意力层交替堆叠。

核心思路：
- Mamba 层处理高效的长距离依赖（O(n) 复杂度）
- Transformer 层提供精确的回忆能力（O(n^2) 但信息无损）
- 两者互补，兼具效率和效果

In [ ]:
class MambaLayer(nn.Module):
    """简化 Mamba 层（用于混合架构）"""
    def __init__(self, d_model, d_state=16, expand_factor=2):
        super().__init__()
        d_inner = d_model * expand_factor
        self.norm = nn.LayerNorm(d_model)
        self.proj_in = nn.Linear(d_model, d_inner, bias=False)
        self.conv = nn.Conv1d(d_inner, d_inner, 4, padding=3, groups=d_inner)
        self.ssm = SelectiveSSM(d_inner, d_state)
        self.proj_out = nn.Linear(d_inner, d_model, bias=False)

    def forward(self, x):
        B, L, D = x.shape
        residual = x
        x = self.norm(x)
        x = self.proj_in(x)
        x = x.transpose(1, 2)
        x = self.conv(x)[:, :, :L]
        x = x.transpose(1, 2)
        x = F.silu(x)
        x = self.ssm(x)
        x = self.proj_out(x)
        return x + residual


# 从 Notebook 03 导入的简化版
class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, dt_min=0.001, dt_max=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        A = torch.arange(1, d_state + 1).float().unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.x_proj = nn.Linear(d_model, d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, d_model, bias=True)

    def forward(self, x):
        B_batch, L, D = x.shape
        N = self.d_state
        A = -torch.exp(self.A_log)
        x_proj = self.x_proj(x)
        B = x_proj[:, :, :N]
        C = x_proj[:, :, N:2*N]
        dt = F.softplus(self.dt_proj(x_proj[:, :, 2*N:2*N+1]))
        dA = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))
        dB = dt.unsqueeze(-1) * B.unsqueeze(2) * A.unsqueeze(0).unsqueeze(0)
        h = torch.zeros(B_batch, D, N, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            h = h * dA[:, t] + dB[:, t] * x[:, t].unsqueeze(-1)
            y = torch.einsum('bdn,bn->bd', h, C[:, t])
            ys.append(y)
        y = torch.stack(ys, dim=1)
        y = y + self.D * x
        return y


class TransformerLayer(nn.Module):
    """简化 Transformer 层"""
    def __init__(self, d_model, n_heads=4, d_ff_mult=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * d_ff_mult),
            nn.GELU(),
            nn.Linear(d_model * d_ff_mult, d_model),
        )

    def forward(self, x, mask=None):
        residual = x
        x = self.norm1(x)
        L = x.size(1)
        if mask is None:
            mask = torch.tril(torch.ones(L, L, device=x.device)).bool()
        x, _ = self.attn(x, x, x, attn_mask=mask)
        x = residual + x
        x = x + self.ffn(self.norm2(x))
        return x


class JambaBlock(nn.Module):
    """Jamba 风格混合层
    
    pattern: [Mamba, Mamba, Mamba, Attention] 重复
    """
    def __init__(self, d_model, n_mamba=3, n_heads=4, d_state=16):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(n_mamba):
            self.layers.append(MambaLayer(d_model, d_state))
        self.layers.append(TransformerLayer(d_model, n_heads))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class JambaModel(nn.Module):
    """Jamba 风格混合模型"""
    def __init__(self, vocab_size, d_model=128, n_blocks=2, n_mamba_per_block=3, n_heads=4, d_state=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            JambaBlock(d_model, n_mamba_per_block, n_heads, d_state)
            for _ in range(n_blocks)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x)


# 验证
jamba = JambaModel(vocab_size=100, d_model=64, n_blocks=2, n_mamba_per_block=2, n_heads=4).to(device)
x = torch.randint(0, 100, (2, 32), device=device)
out = jamba(x)
total = sum(p.numel() for p in jamba.parameters())
print(f'Jamba Model: {x.shape} -> {out.shape}')
print(f'总参数量: {total:,}')
print(f'层结构: x2 [Mamba -> Mamba -> Attention]')

---
## 3. RWKV: 线性 RNN

RWKV (RNN with Werbos-Window-Key-Value) 是另一种替代 Transformer 的方案。

核心思路：
- 将注意力改写为 RNN 形式（token-shift + 线性注意力 + 通道混合）
- 训练时可以像 Transformer 一样并行
- 推理时像 RNN 一样恒定内存

In [ ]:
class WKV(nn.Module):
    """RWKV 的核心: Weighted Key-Value 记忆
    
    简化版本，展示 RWKV 的核心线性注意力机制。
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # 可学习的衰减因子
        self.time_decay = nn.Parameter(torch.randn(n_heads, self.d_head) * 0.1)
        self.time_first = nn.Parameter(torch.randn(n_heads, self.d_head) * 0.1)

    def forward(self, R, K, V):
        """
        R: receptance (类似 Query) [B, L, H, D]
        K: key [B, L, H, D]
        V: value [B, L, H, D]
        """
        B, L, H, D = R.shape
        output = torch.zeros_like(V)

        # 累积状态
        wkv_state = torch.zeros(B, H, D, device=R.device)
        w_state = torch.zeros(B, H, D, device=R.device)

        decay = torch.exp(-torch.exp(self.time_decay))  # [H, D] 衰减因子

        for t in range(L):
            kt = K[:, t]  # [B, H, D]
            vt = V[:, t]  # [B, H, D]
            rt = R[:, t]  # [B, H, D]

            # WKV 计算
            wkv = (wkv_state + torch.exp(self.time_first) * kt * vt) / \
                  (w_state + torch.exp(self.time_first) * kt + 1e-8)

            output[:, t] = rt * wkv

            # 更新状态
            wkv_state = decay * wkv_state + kt * vt
            w_state = decay * w_state + kt

        return output


class RWKVBlock(nn.Module):
    """简化 RWKV Block"""
    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Time mixing (注意力替代)
        self.R_proj = nn.Linear(d_model, d_model, bias=False)
        self.K_proj = nn.Linear(d_model, d_model, bias=False)
        self.V_proj = nn.Linear(d_model, d_model, bias=False)
        self.wkv = WKV(d_model, n_heads)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        # Channel mixing (FFN 替代)
        self.ch_proj1 = nn.Linear(d_model, d_model * 4, bias=False)
        self.ch_proj2 = nn.Linear(d_model * 4, d_model, bias=False)

    def forward(self, x):
        B, L, D = x.shape
        H = self.n_heads
        d_h = self.d_head

        # Time mixing
        residual = x
        x_norm = self.norm1(x)
        R = self.R_proj(x_norm).view(B, L, H, d_h)
        K = self.K_proj(x_norm).view(B, L, H, d_h)
        V = self.V_proj(x_norm).view(B, L, H, d_h)
        wkv_out = self.wkv(R, K, V).view(B, L, D)
        x = residual + self.out_proj(wkv_out)

        # Channel mixing
        residual = x
        x_norm = self.norm2(x)
        x = residual + self.ch_proj2(F.silu(self.ch_proj1(x_norm)))
        return x


# 验证
rwkv = RWKVBlock(d_model=64, n_heads=4).to(device)
x = torch.randn(2, 16, 64, device=device)
out = rwkv(x)
print(f'RWKV Block: {x.shape} -> {out.shape}')
print(f'参数量: {sum(p.numel() for p in rwkv.parameters()):,}')

---
## 4. 横向对比实验

对比 Transformer / Mamba / Jamba(混合) 在相同参数量下的训练速度和效果。

In [ ]:
# 简单序列建模任务：记忆和复制
# 输入: 一段随机 token 序列后跟分隔符
# 目标: 预测下一个 token

def create_copy_task_batch(batch_size, seq_len, vocab_size=20, sep_token=0):
    """创建复制任务数据"""
    x = torch.randint(1, vocab_size, (batch_size, seq_len))
    y = torch.cat([x[:, 1:], torch.randint(1, vocab_size, (batch_size, 1))], dim=1)
    return x, y


# 定义三个对比模型
vocab = 32
d_model = 64

# 1. Transformer (2层)
class TinyTransformer(nn.Module):
    def __init__(self, vocab, d_model, n_layers=2, n_heads=4):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        pe = torch.zeros(512, d_model)
        pos = torch.arange(512).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.layers = nn.ModuleList([
            TransformerLayer(d_model, n_heads) for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, vocab)

    def forward(self, x):
        x = self.emb(x) + self.pe[:, :x.size(1)]
        for layer in self.layers:
            x = layer(x)
        return self.head(x)


# 2. Mamba (2层)
class TinyMamba(nn.Module):
    def __init__(self, vocab, d_model, n_layers=2, d_state=8):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.layers = nn.ModuleList([
            MambaLayer(d_model, d_state) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, x):
        x = self.emb(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))


models = {
    'Transformer': TinyTransformer(vocab, d_model).to(device),
    'Mamba': TinyMamba(vocab, d_model).to(device),
}

# 参数量对比
print('=== 模型参数量 ===')
for name, m in models.items():
    params = sum(p.numel() for p in m.parameters())
    print(f'{name:>15}: {params:>8,} 参数')
print()

In [ ]:
# 训练对比
results = {}
seq_len = 32
batch_size = 32
n_steps = 300

for name, model in models.items():
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    losses = []
    times = []

    model.train()
    for step in range(n_steps):
        x, y = create_copy_task_batch(batch_size, seq_len, vocab)
        x, y = x.to(device), y.to(device)

        start = time.time()
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, vocab), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        elapsed = time.time() - start

        losses.append(loss.item())
        times.append(elapsed)

    results[name] = {'losses': losses, 'times': times}
    avg_time = np.mean(times) * 1000
    print(f'{name:>15}: 最终 Loss={losses[-1]:.4f}, 平均每步 {avg_time:.1f}ms')

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for name, res in results.items():
    axes[0].plot(res['losses'], label=name)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()

for name, res in results.items():
    avg = np.mean(res['times']) * 1000
    axes[1].bar(name, avg)
axes[1].set_ylabel('Time per step (ms)')
axes[1].set_title('Training Speed Comparison')

plt.tight_layout()
plt.show()

In [ ]:
# 序列长度扩展性测试
print('=== 序列长度扩展性 ===')
print(f'{"SeqLen":>8}', end='')
for name in models:
    print(f' {name+" (ms)":>16}', end='')
print()
print('-' * 50)

for seq_len in [32, 64, 128, 256]:
    print(f'{seq_len:>8}', end='')
    x = torch.randint(0, vocab, (4, seq_len), device=device)
    for name, model in models.items():
        model.eval()
        times = []
        with torch.no_grad():
            for _ in range(5):
                start = time.time()
                _ = model(x)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                times.append(time.time() - start)
        avg = np.mean(times) * 1000
        print(f' {avg:>14.1f}', end='')
    print()

print('\n观察: Transformer 时间随序列长度二次增长，Mamba 线性增长')

---
## 总结: 架构选择指南

| 维度 | Transformer | Mamba | Mamba-2 | Jamba (混合) | RWKV |
|------|------------|-------|---------|-------------|------|
| 训练复杂度 | O(n²) | O(n) | O(n) | O(n) ~ O(n²) | O(n) |
| 推理内存 | O(n) KV-Cache | O(1) 恒定 | O(1) 恒定 | 混合 | O(1) 恒定 |
| 长序列 | 受限 | 优秀 | 优秀 | 优秀 | 优秀 |
| 精确回忆 | 最强 | 中等 | 中等 | 强 | 中等 |
| 生态成熟度 | 最高 | 中等 | 中等 | 较新 | 中等 |
| 代表模型 | GPT-4, LLaMA | Mamba | Mamba-2 | Jamba | RWKV-6 |

### 选择建议
- **通用 LLM**: Transformer 仍是首选（生态成熟、效果稳定）
- **长序列任务**: Mamba / Mamba-2（DNA建模、长文档、时序预测）
- **追求极致推理效率**: Mamba-2 或 RWKV
- **兼顾效率和精确回忆**: Jamba（Mamba+Transformer 混合）
- **研究前沿**: 关注 Mamba-2 的结构化注意力框架，它统一了 SSM 和注意力